# Comparing two conditions — `shapece` differential analysis & visualization

This notebook takes **normalized reactivity profiles with replicates** for two conditions
(e.g. two temperatures, ± ligand, ± protein) and answers:

1. **Did the probe work?** (a mandatory QC gate)
2. **Are the replicates reproducible?**
3. **Which regions changed?** — deltaSHAPE (Smola et al. 2015)
4. **Which individual nucleotides changed?** — t-test + FDR
5. **Did anything change globally?** — exact permutation test

…and renders publication-ready **skyline, arc, and circle** plots.

**Prerequisite:** run `SHAPE_CE_Analysis.ipynb` first to produce reactivity profiles, or load
your own. Nothing here is specific to any organism, probe, or hypothesis.

**How to use:** run cells top to bottom. You edit only the `CONFIG` cell in Step 1.


## Step 0 — Install and import

In [ ]:
try:
    import google.colab  # noqa
    !pip -q install ViennaRNA openpyxl pandas scipy matplotlib ipywidgets
    !pip -q install git+https://github.com/johnnythor-micro/shape-ce.git
except ImportError:
    pass

import numpy as np, matplotlib.pyplot as plt
import shapece as sc
from shapece import stats, plots, ui
print("shapece", sc.__version__)

## Step 1 — 🔧 Set up the comparison

Two boxes appear: **condition labels** and (optionally) your **RNA sequence** for the structure
plots in Step 7. No code editing required.

Then load a **replicate matrix** per condition — shape `(n_replicates, n_nucleotides)` of
*normalized* reactivity, with `np.nan` for no-data positions. At least 2 replicates per condition
(the per-nucleotide error is the SEM across replicates); 3+ strongly recommended.

If you ran `SHAPE_CE_Analysis.ipynb`, these are the arrays in its `profiles` dict.

In [ ]:
import ipywidgets as w
from IPython.display import display

label1 = w.Text(value="condition 1", description="label 1:")
label2 = w.Text(value="condition 2", description="label 2:")
mask5  = w.BoundedIntText(value=0, min=0, max=500, description="mask 5':")
mask3  = w.BoundedIntText(value=0, min=0, max=500, description="mask 3':")
seq_box = ui.SequenceInput(title="RNA sequence (optional — for the structure plots)",
                           hint="Leave empty to skip Step 7.")
display(w.VBox([w.HBox([label1, label2]), w.HBox([mask5, mask3])]))
seq_box.display()

In [ ]:
CONFIG = {"label1": label1.value, "label2": label2.value,
          "sequence": seq_box.value or None,
          "mask5": mask5.value, "mask3": mask3.value}

# --- load your replicate matrices here, shape (n_reps, n_nt) ---
# reps1 = np.loadtxt("cond1.csv", delimiter=","); reps2 = np.loadtxt("cond2.csv", delimiter=",")
rng = np.random.RandomState(0); n = 120                    # DEMO DATA — replace with yours
base = np.abs(rng.normal(0.4, 0.3, n))
reps1 = np.array([base + rng.normal(0, 0.05, n) for _ in range(3)])
hot = base.copy(); hot[60:67] += 1.0
reps2 = np.array([hot + rng.normal(0, 0.05, n) for _ in range(3)])

print(f"{CONFIG['label1']}: {reps1.shape} | {CONFIG['label2']}: {reps2.shape}")

## Step 2 — QC gate: did the probe actually work?  **(do not skip)**

If you have the raw (+) and (−) **peak areas**, run this. When probe signal is weak the (+) and
(−) lanes are nearly identical, background subtraction cannot isolate reactivity, and what
survives is the natural-stop pattern — which is *highly reproducible* and easily mistaken for
clean data showing "no difference".

**A reproducible background is not a measurement.** If this fails, stop and repeat the
experiment with fresh reagent.

In [ ]:
# Supply peak areas from the main analysis notebook, if available:
# qc = stats.probe_signal_qc(area_rx, area_bg)
# print("passed:", qc["passed"]); print(qc)
#
# Healthy:  rx_bg_correlation < ~0.95 ; mean_area_ratio > 1 ; background_carryover ~ 0
print("Skipping: supply area_rx / area_bg from your peak quantification to enable this gate.")

## Step 3 — Replicate reproducibility

`loo_r` is each replicate against the mean of the others. Good CE-SHAPE replicates reach
r ≈ 0.8–0.9. Investigate any outlier **before** comparing conditions.

In [ ]:
for label, reps in [(CONFIG["label1"], reps1), (CONFIG["label2"], reps2)]:
    rs = stats.replicate_stats(reps)
    print(f"{label}: loo_r = {np.round(rs['loo_r'], 3)}  pairwise_r = {np.round(rs['pairwise_r'], 3)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
plots.linreg(reps1[0], reps1[1], ax=ax[0], labels=("rep 1", "rep 2"))
plots.heatmap(np.vstack([reps1, reps2]), ax=ax[1],
              row_labels=[f"{CONFIG['label1']}·{i+1}" for i in range(len(reps1))] +
                         [f"{CONFIG['label2']}·{i+1}" for i in range(len(reps2))])
plt.tight_layout(); plt.show()

## Step 4 — Overlay the two conditions (skyline plot)

A **skyline** draws each nucleotide as a full-width step, so two profiles superimpose without
hiding each other. Where the traces separate, reactivity changed.

In [ ]:
m1, e1 = stats.sem_from_replicates(reps1)
m2, e2 = stats.sem_from_replicates(reps2)

fig, ax = plt.subplots(2, 1, figsize=(14, 6))
plots.skyline({CONFIG["label1"]: m1, CONFIG["label2"]: m2}, ax=ax[0],
              title="conditions overlaid")
plots.profile(m1, error=e1, ax=ax[1], title=f"{CONFIG['label1']} (mean ± SEM)")
plt.tight_layout(); plt.show()

## Step 5 — deltaSHAPE: which *regions* changed?

Smooths reactivities and errors (3-nt window), computes a **Z-factor** (>0 = the 95% confidence
intervals don't overlap) and a **standard score** (how big is this change relative to all other
changes), then calls a **site** where ≥3 nucleotides in a 5-nt window pass both.

Requiring changes to cluster is what gives deltaSHAPE its power: real rearrangements affect
adjacent nucleotides; noise does not.

Sign convention: positive = **more reactive in condition 1**.

In [ ]:
res = stats.delta_shape(None, None, replicates1=reps1, replicates2=reps2,
                        pad=1, mask5=CONFIG["mask5"], mask3=CONFIG["mask3"])
print(f"{res['n_sites']} site(s) called\n")
for s in res["sites"]:
    print(f"  nt {s['start']}-{s['end']}  {s['direction']:8}  mean Δ = {s['mean_diff']:+.2f}  ({s['n_nt']} nt)")

plots.delta_shape_plot(res, labels=(CONFIG["label1"], CONFIG["label2"]))
plt.tight_layout(); plt.show()

## Step 6 — Per-nucleotide testing (t-test + FDR) and the global test

These answer different questions from deltaSHAPE and are complementary.

> ⚠️ **Power.** With n = 3 and hundreds of nucleotides, the per-nucleotide test is usually
> underpowered after FDR correction — zero hits is common *even when a real difference exists*.
> And with 3 vs 3 replicates the permutation test can never return p below 0.10.
> Report these limits honestly rather than over-reading a null.

In [ ]:
tt = stats.ttest_fdr(reps1, reps2, min_delta=0.3, q_thresh=0.10)
print(f"per-nucleotide: {tt['n_significant']} significant (q<0.10 and |Δ|>=0.3)")

pt = stats.permutation_test(reps1, reps2)
print(f"global permutation: p = {pt['p_value']:.2f} "
      f"(minimum attainable p = {pt['min_attainable_p']:.2f}, {pt['n_partitions']} partitions)")

## Step 7 — Map reactivity onto structure (arc and circle plots)

**Arc plot**: pairs as arcs over the sequence. Nested arcs = helices; crossing arcs = pseudoknots.
In a good model, red (reactive) nucleotides fall in the gaps between arcs. Two structures can be
mirrored above/below to compare them.

**Circle plot**: sequence on a circle, pairs as chords. Every contact is the same visual scale,
so **long-range interactions stay legible** — use this for long RNAs where arcs become unreadable.

Requires a sequence in `CONFIG`. Folding uses ViennaRNA (Deigan SHAPE method).

In [ ]:
seq = CONFIG["sequence"]
if seq:
    ss_shape, mfe_s = sc.structure.fold(seq, m1, method="deigan")
    ss_thermo, mfe_t = sc.structure.fold(seq, None)
    print(f"SHAPE-directed {mfe_s:.1f} kcal/mol | thermodynamic {mfe_t:.1f} kcal/mol")

    fig, ax = plt.subplots(figsize=(14, 5))
    plots.arc(ss_shape, reactivity=m1, sequence=seq, structure2=ss_thermo, ax=ax,
              labels=("SHAPE-directed", "thermodynamic"), title="arc: two models compared")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(8, 8))
    plots.circle(ss_shape, reactivity=m1, sequence_length=len(seq), ax=ax,
                 title="circle: long-range contacts")
    plt.tight_layout(); plt.show()
else:
    print("Set CONFIG['sequence'] to enable structure plots.")

## Step 8 — Export

Writes a Prism-ready workbook (replicates + per-condition mean/SD) and a deltaSHAPE site table.

For statistics in Prism instead of here, see **`docs/STATISTICS_GUIDE.md`**.
For the methods behind this notebook, see **`docs/DIFFERENTIAL_ANALYSIS.md`** and
**`docs/VISUALIZATION_GUIDE.md`**.

In [ ]:
import pandas as pd
from shapece import report

names1 = [f"{CONFIG['label1']}_rep{i+1}" for i in range(len(reps1))]
names2 = [f"{CONFIG['label2']}_rep{i+1}" for i in range(len(reps2))]
nt = np.arange(1, reps1.shape[1] + 1)
base = list(CONFIG["sequence"]) if CONFIG["sequence"] else ["N"] * len(nt)

df = report.reactivity_table(nt, base, dict(zip(names1 + names2, list(reps1) + list(reps2))))
df = report.add_group_stats(df, {CONFIG["label1"]: names1, CONFIG["label2"]: names2})
df["delta_smoothed"] = res["smoothed_diff"]
df["z_factor"] = res["z_factors"]
df["z_score"] = res["z_scores"]
df["deltaSHAPE_significant"] = res["significant"].astype(int)
report.write_excel_for_prism("comparison_for_prism.xlsx", df,
    notes="deltaSHAPE columns included. Sign: positive = more reactive in " + CONFIG["label1"])

pd.DataFrame(res["sites"]).to_csv("deltashape_sites.csv", index=False)
print("wrote comparison_for_prism.xlsx and deltashape_sites.csv")
df.head()